# 01 — Your First Agent

**What you'll learn:**
- What is Microsoft Agent Framework and why it exists
- How to create an agent backed by Azure OpenAI
- Non-streaming vs streaming responses
- Inspecting the `AgentResponse` object

---

## What is Microsoft Agent Framework?

Microsoft Agent Framework is the **successor to both Semantic Kernel and AutoGen**. It combines:

- **AutoGen's** simple agent abstractions for single- and multi-agent patterns
- **Semantic Kernel's** enterprise features: sessions, type safety, middleware, telemetry
- **New**: Graph-based workflows for explicit multi-agent orchestration

It supports Azure OpenAI, OpenAI, Anthropic, Ollama, and more — but in this course
we'll use **Azure AI Foundry** throughout.

### Two Primary Capabilities

| Capability | Description |
|-----------|-------------|
| **Agents** | Individual agents that use LLMs, call tools, and generate responses |
| **Workflows** | Graph-based workflows connecting agents and functions for multi-step tasks |

## Setup

Every notebook starts with the same setup block. We load environment variables
and create a client using `DefaultAzureCredential` (which uses your `az login` session).

> **First time?** Make sure you've run:
> ```bash
> cp .env.example .env   # then edit .env with your deployment name
> uv sync
> az login
> ```

In [19]:
import os

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv

from agent_framework.azure import AzureOpenAIResponsesClient

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"]

## Creating an Agent

An **Agent** needs two things:
1. A **client** — the connection to the LLM service
2. **Instructions** — the system prompt that shapes the agent's behavior

The `AzureOpenAIResponsesClient` connects to Azure AI Foundry using the
Responses API. We call `.as_agent()` to turn it into an agent.

In [20]:
credential = DefaultAzureCredential()

client = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    credential=credential,
)

agent = client.as_agent(
    name="InsuranceAdvisor",
    instructions=(
        "You are a knowledgeable insurance advisor. "
        "Explain insurance concepts clearly and concisely, "
        "using simple language that anyone can understand. "
        "Keep answers to 2-3 short paragraphs."
    ),
)

print(f"Agent created: {agent.name}")

Agent created: InsuranceAdvisor


## Non-Streaming Response

The simplest way to use an agent — call `agent.run()` and wait for the
complete response. Good for back-end processing or when you need the full
answer before doing anything with it.

In [21]:
result = await agent.run("What are the main types of life insurance policies?")

print(result)

Life insurance comes in a few main types, each designed to meet different needs. The two primary categories are **term life insurance** and **permanent life insurance**.

1. **Term Life Insurance**: This type of policy provides coverage for a specific period, or "term," such as 10, 20, or 30 years. If you pass away during the term, the insurance pays a death benefit to your beneficiaries. Term life is typically more affordable but doesn’t build cash value, and coverage ends when the term expires.

2. **Permanent Life Insurance**: This category includes several policy types, such as **whole life** and **universal life** insurance. Permanent policies provide lifelong coverage and have a cash value component that can grow over time. The cash value can be used during your lifetime, either through loans or withdrawals. Whole life insurance has fixed premiums and guaranteed growth, while universal life offers more flexibility in premiums and coverage.

Choosing between these depends on your 

## Streaming Response

For a chat-like UX, stream tokens as they arrive. Pass `stream=True`
and iterate over the chunks. Each chunk has a `.text` attribute — it may
be `None` for non-text chunks (tool calls, metadata), so we check first.

In [22]:
print("Agent: ", end="", flush=True)

async for chunk in agent.run(
    "Explain the difference between term and whole life insurance.",
    stream=True,
):
    if chunk.text:
        print(chunk.text, end="", flush=True)

print()  # newline at the end

Agent: Term life insurance and whole life insurance are two common types of life insurance, but they work differently.

**Term life insurance** provides coverage for a specific period of time, such as 10, 20, or 30 years. If you pass away during that term, your beneficiary receives the death benefit (the payout). However, if you outlive the policy, the coverage ends, and there’s no payout or cash value. Term life insurance is typically less expensive because it’s focused solely on providing death benefit protection for a set time.

**Whole life insurance**, on the other hand, covers you for your entire life as long as you keep paying premiums. It also includes a savings component called "cash value," which grows over time and can be accessed or borrowed against while you’re alive. Whole life insurance is more expensive because it lasts a lifetime and includes this additional cash value feature.

Essentially, term life is ideal if you need affordable coverage for a specific time frame, 

## Inspecting the AgentResponse

The `AgentResponse` returned by `agent.run()` isn't just a string — it's a
rich object. Let's explore what's inside.

In [23]:
result = await agent.run(
    "What is an insurance deductible? Give me a one-sentence answer."
)

# The full text of the response
print("=== .text ===")
print(result.text)
print()

# The list of messages in the response
print("=== .messages ===")
for msg in result.messages:
    print(f"  role={msg.role}, text={msg.text[:80]}...")

=== .text ===
An insurance deductible is the amount of money you agree to pay out of pocket for a covered claim before your insurance company starts paying its share.

=== .messages ===
  role=assistant, text=An insurance deductible is the amount of money you agree to pay out of pocket fo...


## Clean Up

When using `DefaultAzureCredential` as an async context, close it when done.

In [24]:
await credential.close()
print("Done!")

Done!


---

## Key Takeaways

- **Agent = Client + Instructions** — that's all you need to get started
- `agent.run(prompt)` → complete response (`AgentResponse`)
- `agent.run(prompt, stream=True)` → async iterator of `AgentResponseUpdate` chunks
- The framework handles retries, auth, and message formatting for you

**Next up:** [02 — Function Tools](./02-function-tools.ipynb) — give your agent
callable tools like premium calculators and policy lookups.